In [5]:
!pip install pymupdf
!pip install langchain
!pip install langchain-openai
!pip install langchain-chroma
!pip install chromadb
!pip install python-dotenv
!pip install streamlit


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
print(api_key is not None)

False


In [7]:
import fitz
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_path = "data/sample1.pdf"

doc = fitz.open(pdf_path)

# Store text separately for each page
pages = []

for page_number, page in enumerate(doc, start=1):
    page_text = page.get_text().strip()

    if page_text:
        pages.append({
            "text": page_text,
            "page": page_number,
            "source": pdf_path
        })

print("Total pages with text:", len(pages))

c:\Users\yadav\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total pages with text: 2


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []
metadatas = []

for page in pages:

    page_chunks = text_splitter.split_text(page["text"])

    for chunk in page_chunks:
        chunks.append(chunk)

        metadatas.append({
            "source": page["source"],
            "page": page["page"]
        })

print("Total chunks:", len(chunks))

Total chunks: 4


In [10]:
print("First chunk:")
print(chunks[0])

print("\nMetadata:")
print(metadatas[0])

First chunk:
ePublishing System
Published Date: 28-Sep-2026
Basic Details
Organisation Chain
Indian Rare Earths Limited
Tender Reference
Number
GEM/2026/B/7653569
Tender ID
2026_IRELD_841521_1
Withdrawal Allowed
Yes
Tender Type
Open Tender
Form Of Contract
Buy
Tender Category
Goods
No. of Covers
2
General Tech Evaluation
No
ItemWise Tech
Evaluation
No
Payment Mode
Offline
Multi Currency Allowed
(BOQ)
No
Multi Currency Allowed
(Fee)
No
Two Stage Bidding
No
Payment Instruments
Offline
S.No
Instrument Type
1
RTGS / ECS / NEFT
2
As Per Tender Document
Covers Information (No. Of Covers - 2)
Cover
No
Cover Type
Description
Type
1
Fee/PreQual/Technical
Procurement of
LED Street
Light
.pdf
2
Finance
Procurement of
LED Street
Light
.xls

Metadata:
{'source': 'data/sample1.pdf', 'page': 1}


In [11]:
print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- CHUNK {i+1} ---\n")
    print(chunk[:500])

Number of chunks: 4

--- CHUNK 1 ---

ePublishing System
Published Date: 28-Sep-2026
Basic Details
Organisation Chain
Indian Rare Earths Limited
Tender Reference
Number
GEM/2026/B/7653569
Tender ID
2026_IRELD_841521_1
Withdrawal Allowed
Yes
Tender Type
Open Tender
Form Of Contract
Buy
Tender Category
Goods
No. of Covers
2
General Tech Evaluation
No
ItemWise Tech
Evaluation
No
Payment Mode
Offline
Multi Currency Allowed
(BOQ)
No
Multi Currency Allowed
(Fee)
No
Two Stage Bidding
No
Payment Instruments
Offline
S.No
Instrument Type
1
RT

--- CHUNK 2 ---

Cover
No
Cover Type
Description
Type
1
Fee/PreQual/Technical
Procurement of
LED Street
Light
.pdf
2
Finance
Procurement of
LED Street
Light
.xls
Notice: Document download date is over. Documents can no longer be downloaded.
Fee Details
Tender Fee Details [Total Fee in ₹: 1]
Tender Fee in ₹
1
Fee Payable To
ICAR Unit CSSRI
Fee Payable At
Karnal
Tender Fee
Exemption Allowed
Yes
EMD Fee Details
EMD Amount in ₹
25,025
EMD Exemption
Allowed
Yes

In [12]:
from sentence_transformers import SentenceTransformer

In [13]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3744.88it/s]


Embedding model loaded!


In [14]:
test_embedding = embedding_model.encode(chunks[0])

print("Embedding length:", len(test_embedding))
print("First 10 values:", test_embedding[:10])

Embedding length: 384
First 10 values: [-0.04145869  0.03556165 -0.01675627  0.01792394 -0.02507868 -0.04178052
 -0.01144013  0.05706248 -0.06537346 -0.00277812]


In [15]:
import chromadb

In [16]:
client = chromadb.PersistentClient(path="./chroma_db")

improved_collection = client.get_or_create_collection(
    name="pdf_documents_v2"
)

print("Collection ready:", improved_collection.name)

Collection ready: pdf_documents_v2


In [17]:
chunk_embeddings = embedding_model.encode(chunks)

print("Number of embeddings:", len(chunk_embeddings))
print("Embedding size:", len(chunk_embeddings[0]))

Number of embeddings: 4
Embedding size: 384


In [18]:
improved_collection.add(
    ids=[f"sample1_chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),
    metadatas=metadatas
)

print("Documents stored:", improved_collection.count())

Documents stored: 4


In [19]:
query = "What is the tender reference number?"

query_embedding = embedding_model.encode(query).tolist()

results = improved_collection.query(
    query_embeddings=[query_embedding],
    n_results=4
)

for i, document in enumerate(results["documents"][0]):
    print(f"\n--- Result {i+1} ---")
    print(document)
    print("Metadata:", results["metadatas"][0][i])


--- Result 1 ---
ePublishing System
Published Date: 28-Sep-2026
Basic Details
Organisation Chain
Indian Rare Earths Limited
Tender Reference
Number
GEM/2026/B/7653569
Tender ID
2026_IRELD_841521_1
Withdrawal Allowed
Yes
Tender Type
Open Tender
Form Of Contract
Buy
Tender Category
Goods
No. of Covers
2
General Tech Evaluation
No
ItemWise Tech
Evaluation
No
Payment Mode
Offline
Multi Currency Allowed
(BOQ)
No
Multi Currency Allowed
(Fee)
No
Two Stage Bidding
No
Payment Instruments
Offline
S.No
Instrument Type
1
RTGS / ECS / NEFT
2
As Per Tender Document
Covers Information (No. Of Covers - 2)
Cover
No
Cover Type
Description
Type
1
Fee/PreQual/Technical
Procurement of
LED Street
Light
.pdf
2
Finance
Procurement of
LED Street
Light
.xls
Metadata: {'page': 1, 'source': 'data/sample1.pdf'}

--- Result 2 ---
Doc Download / Sale
Start Date
13-Jun-2026 11:00 AM
Doc Download / Sale
End Date
02-Jul-2026 03:00 PM
Clarification Start Date
13-Jun-2026 11:00 AM
Clarification End Date
02-Jul-2026 03:0

In [20]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Reranker loaded successfully")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 4199.54it/s]


Reranker loaded successfully


In [21]:
question = "What is the tender reference number?"

results = improved_collection.query(
    query_embeddings=[embedding_model.encode(question).tolist()],
    n_results=4
)

documents = results["documents"][0]

pairs = [[question, document] for document in documents]

scores = reranker.predict(pairs)

for i, score in enumerate(scores):
    print(f"Result {i+1}: {score:.4f}")

Result 1: 3.2600
Result 2: -7.7718
Result 3: -7.4161
Result 4: -9.5073


In [22]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

api_key = os.getenv("OPENAI_API_KEY")

print("API key loaded:", bool(api_key))
print("Key ending:", api_key[-6:] if api_key else None)

API key loaded: False
Key ending: None


In [23]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import ollama

In [25]:
import ollama

response = ollama.chat(
    model="llama3.2:1b",
    messages=[
        {"role": "user", "content": "Say hello in one sentence."}
    ],
    options={"num_ctx": 512},
    keep_alive=0
)

print(response["message"]["content"])

Hello.


In [39]:
def ask_pdf_v2(question, n_candidates=4, n_final=2):

    # 1. Convert question to embedding
    query_embedding = embedding_model.encode(question).tolist()

    # 2. Retrieve candidate chunks from Chroma
    results = improved_collection.query(
        query_embeddings=[query_embedding],
        n_results=n_candidates
    )

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    # 3. Rerank candidates
    pairs = [[question, document] for document in documents]
    scores = reranker.predict(pairs)

    # 4. Sort by reranker score
    ranked = sorted(
        zip(documents, metadatas, scores),
        key=lambda x: x[2],
        reverse=True
    )

    # 5. Keep the best chunks
    top_results = ranked[:n_final]

    # 6. Build context with source information
    context_parts = []

    for document, metadata, score in top_results:
        context_parts.append(
            f"""
Source: {metadata['source']}
Page: {metadata['page']}

Content:
{document}
"""
        )

    context = "\n\n".join(context_parts)

    # 7. Create grounded prompt
    prompt = f"""
You are a document-grounded question answering assistant.

Answer the question using ONLY the information provided in the context.

Rules:
1. Do not use outside knowledge.
2. Do not invent information.
3. If the answer is not present in the context, say:
   "I could not find sufficient information in the provided document."
4. Prefer exact names, numbers and references from the document.
5. Give the source and page number for the information used.

Context:
{context}

Question:
{question}

Answer:
"""

    # 8. Generate answer
    response = ollama.chat(
        model="llama3.2:1b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        options={"num_ctx": 1024}
    )

    return response["message"]["content"]

In [81]:
import re

def extract_structured_answer(question, document):
    """
    Extract exact values from common structured fields.
    Returns None if the question is not a recognized structured field.
    """

    question_lower = question.lower()

    # Normalize whitespace so:
    # Tender Reference
    # Number
    # becomes:
    # Tender Reference Number
    text = re.sub(r"\s+", " ", document).strip()

    # Tender Reference Number
    if "tender reference" in question_lower:
        match = re.search(
            r"Tender\s+Reference\s+Number\s+([A-Za-z0-9./_-]+)",
            text,
            re.IGNORECASE
        )

        if match:
            return match.group(1)

    # Tender ID
    if "tender id" in question_lower:
        match = re.search(
            r"Tender\s+ID\s+([A-Za-z0-9./_-]+)",
            text,
            re.IGNORECASE
        )

        if match:
            return match.group(1)

    # Published Date
    if "published date" in question_lower:
        match = re.search(
            r"Published\s+Date\s*:?\s*([0-9]{1,2}-[A-Za-z]{3}-[0-9]{4}(?:\s+[0-9]{1,2}:[0-9]{2}\s*(?:AM|PM))?)",
            text,
            re.IGNORECASE
        )

        if match:
            return match.group(1)

    # Organisation
    if "organisation" in question_lower or "organization" in question_lower:
        match = re.search(
            r"Organisation\s+Chain\s+(.+?)\s+Tender\s+Reference\s+Number",
            text,
            re.IGNORECASE
        )

        if match:
            return match.group(1).strip()
        
    # Form of Contract
    if "form of contract" in question_lower:
        match = re.search(
            r"Form\s+Of\s+Contract\s+([A-Za-z]+)",
            text,
            re.IGNORECASE
        )

        if match:
            return match.group(1)
        
        # Tender Value
    if "tender value" in question_lower:
        match = re.search(
            r"Tender\s+Value\s+in\s+₹?\s*([0-9,]+)",
            text,
            re.IGNORECASE
        )

        if match:
            return "₹" + match.group(1)

            # Product Category
    if "product category" in question_lower:
        match = re.search(
            r"Product\s+Category\s+(.+?)\s+Sub\s+category",
            text,
            re.IGNORECASE
        )

        if match:
            return match.group(1).strip()

        # Contract Type
    if "contract type" in question_lower:
        match = re.search(
            r"Contract\s+Type\s+([A-Za-z]+)",
            text,
            re.IGNORECASE
        )

        if match:
            return match.group(1)

    return None

In [82]:
print(
    extract_structured_answer(
        "What is the tender value?",
        best_chunk
    )
)

None


In [46]:
best_chunk = results["documents"][0][0]

print(
    extract_structured_answer(
        "What is the tender reference number?",
        best_chunk
    )
)

GEM/2026/B/7653569


In [47]:
print(
    extract_structured_answer(
        "What is the tender ID?",
        best_chunk
    )
)

2026_IRELD_841521_1


In [60]:
print(
    extract_structured_answer(
        "What is the published date of the tender?",
        best_chunk
    )
)

28-Sep-2026


In [49]:
print(
    extract_structured_answer(
        "Which organisation issued the tender?",
        best_chunk
    )
)

Indian Rare Earths Limited


In [40]:
answer = ask_pdf_v2(
    "What is the tender reference number?"
)

print(answer)

According to the provided document, the tender reference number is:

GEM/2026/B/7653569

Source: data/sample1.pdf
Page: 1

This is indicated on page 1 of the document.


In [ ]:
def ask_pdf_v3(question, n_candidates=4):

    # 1. Convert question to embedding
    query_embedding = embedding_model.encode(question).tolist()

    # 2. Retrieve candidates
    results = improved_collection.query(
        query_embeddings=[query_embedding],
        n_results=n_candidates
    )

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    # 3. Rerank candidates
    pairs = [[question, document] for document in documents]
    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(documents, metadatas, scores),
        key=lambda x: x[2],
        reverse=True
    )

    # --------------------------------------------------
    # 4. STRUCTURED EXTRACTION
    # Search all ranked chunks, not just Top 1
    # --------------------------------------------------

    for document, metadata, score in ranked:

        extracted_answer = extract_structured_answer(
            question,
            document
        )

        if extracted_answer:
            return (
                f"{extracted_answer}\n\n"
                f"Source: {metadata['source']}\n"
                f"Page: {metadata['page']}"
            )

    # --------------------------------------------------
    # 5. SEMANTIC QUESTION → LLM
    # --------------------------------------------------

    # Use the best 2 chunks for semantic questions
    top_results = ranked[:2]

    context_parts = []

    for document, metadata, score in top_results:
        context_parts.append(
            f"""
Source: {metadata['source']}
Page: {metadata['page']}

Content:
{document}
"""
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
You are a document-grounded question answering assistant.

Answer the question using ONLY the information provided in the context.

Rules:
1. Do not use outside knowledge.
2. Do not invent information.
3. If the answer is not present, say:
   "I could not find sufficient information in the provided document."
4. Give a concise answer.
5. Do not copy the entire context.
6. Include the source and page.

Context:
{context}

Question:
{question}

Answer:
"""

    response = ollama.chat(
        model="llama3.2:1b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        options={
            "num_ctx": 1024,
            "temperature": 0
        },
        keep_alive=0
    )

    return (
        response["message"]["content"]
        + f"\n\nSource: {ranked[0][1]['source']}, "
          f"Page: {ranked[0][1]['page']}"
    )

In [76]:
questions = [
    "What is the tender reference number?",
    "What is the tender ID?",
    "Which organisation issued the tender?",
    "What is the published date of the tender?"
]

for question in questions:
    print("\n" + "=" * 70)
    print("QUESTION:", question)
    print("ANSWER:")
    print(ask_pdf_v3(question))


QUESTION: What is the tender reference number?
ANSWER:
GEM/2026/B/7653569

Source: data/sample1.pdf
Page: 1

QUESTION: What is the tender ID?
ANSWER:
2026_IRELD_841521_1

Source: data/sample1.pdf
Page: 1

QUESTION: Which organisation issued the tender?
ANSWER:
Indian Rare Earths Limited

Source: data/sample1.pdf
Page: 1

QUESTION: What is the published date of the tender?
ANSWER:
28-Sep-2026

Source: data/sample1.pdf
Page: 1


In [42]:
for question in test_questions:
    print("\n" + "=" * 70)
    print("QUESTION:", question)

    answer = ask_pdf_v2(question)

    print("ANSWER:", answer)


QUESTION: What is the tender reference number?
ANSWER: I could not find sufficient information in the provided document.

Source: data/sample1.pdf
Page: 1

QUESTION: What is the tender ID?
ANSWER: I could not find sufficient information in the provided document.

QUESTION: Which organisation issued the tender?
ANSWER: I could not find sufficient information in the provided document.

QUESTION: What is the published date of the tender?
ANSWER: Page: 1
Source: data/sample1.pdf
Published Date: 28-Sep-2026

I could not find sufficient information in the provided document.

QUESTION: What type of document is this?
ANSWER: The document type is Tendernotice_1.pdf.

Source: data/sample1.pdf
Page: 2

The source and page number for the information used is data/sample1.pdf, page 2.


In [62]:
evaluation_data = [
    {
        "question": "What is the tender reference number?",
        "expected": "GEM/2026/B/7653569"
    },
    {
        "question": "What is the tender ID?",
        "expected": "2026_IRELD_841521_1"
    },
    {
        "question": "Which organisation issued the tender?",
        "expected": "Indian Rare Earths Limited"
    },
    {
        "question": "What is the published date of the tender?",
        "expected": "28-Sep-2026"
    }
]

print("Evaluation questions:", len(evaluation_data))

Evaluation questions: 4


In [63]:
results = []

for item in evaluation_data:

    question = item["question"]
    expected = item["expected"]

    answer = ask_pdf_v3(question)

    results.append({
        "question": question,
        "expected": expected,
        "answer": answer
    })

    print("\n" + "=" * 80)
    print("QUESTION:", question)
    print("EXPECTED:", expected)
    print("ANSWER:", answer)


QUESTION: What is the tender reference number?
EXPECTED: GEM/2026/B/7653569
ANSWER: GEM/2026/B/7653569

Source: data/sample1.pdf
Page: 1

QUESTION: What is the tender ID?
EXPECTED: 2026_IRELD_841521_1
ANSWER: 2026_IRELD_841521_1

Source: data/sample1.pdf
Page: 1

QUESTION: Which organisation issued the tender?
EXPECTED: Indian Rare Earths Limited
ANSWER: Indian Rare Earths Limited

Source: data/sample1.pdf
Page: 1

QUESTION: What is the published date of the tender?
EXPECTED: 28-Sep-2026
ANSWER: 28-Sep-2026

Source: data/sample1.pdf
Page: 1


In [64]:
correct = 0

for result in results:

    answer = result["answer"].lower()
    expected = result["expected"].lower()

    if expected in answer:
        correct += 1

accuracy = (correct / len(results)) * 100

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 100.00%


In [77]:
semantic_questions = [
    "What type of tender is this?",
    "What is the tender category?",
    "What is the form of contract?",
    "Is withdrawal of the tender allowed?",
    "What is the main procurement item?",
    "What is being procured through this tender?",
    "Which organisation is associated with this tender?",
    "What kind of procurement is described in the document?"
]

print("Semantic questions:", len(semantic_questions))

Semantic questions: 8


In [78]:
for question in semantic_questions:
    print("\n" + "=" * 80)
    print("QUESTION:", question)

    answer = ask_pdf_v3(question)

    print("ANSWER:", answer)


QUESTION: What type of tender is this?
ANSWER: This is an Open Tender.

Source: data/sample1.pdf, Page: 1

QUESTION: What is the tender category?
ANSWER: Tender Category
Goods

Source: data/sample1.pdf, Page: 1

QUESTION: What is the form of contract?
ANSWER: Buy

Source: data/sample1.pdf
Page: 1

QUESTION: Is withdrawal of the tender allowed?
ANSWER: Yes

Source: data/sample1.pdf, Page: 1

QUESTION: What is the main procurement item?
ANSWER: The main procurement item is "Procurement of LED Street Light".

Source: data/sample1.pdf, Page: 2

QUESTION: What is being procured through this tender?
ANSWER: LED Street Light

Source: data/sample1.pdf, Page: 1

QUESTION: Which organisation is associated with this tender?
ANSWER: Indian Rare Earths Limited

Source: data/sample1.pdf
Page: 1

QUESTION: What kind of procurement is described in the document?
ANSWER: The procurement of LED Street Light is described as a "Procurement of LED Street Light".

Source: data/sample1.pdf, Page: 1


In [84]:
hard_questions = [
    "What is the tender value?",
    "Where is the tender work located?",
    "What is the bid validity period?",
    "What is the period of work?",
    "What is the product category?",
    "What is the contract type?",
    "What is the pincode of the tender location?",
    "What is being procured and what is its tender value?"
]

print("Hard questions:", len(hard_questions))

Hard questions: 8


In [80]:
for question in hard_questions:
    print("\n" + "=" * 80)
    print("QUESTION:", question)

    answer = ask_pdf_v3(question)

    print("ANSWER:", answer)


QUESTION: What is the tender value?
ANSWER: I could not find sufficient information in the provided document.

Source: data/sample1.pdf, Page: 2

QUESTION: Where is the tender work located?
ANSWER: Manavalakurichi

Source: data/sample1.pdf, Page: 2

QUESTION: What is the bid validity period?
ANSWER: Bid Validity (Days): 90

Source: data/sample1.pdf, Page: 2

QUESTION: What is the period of work?
ANSWER: 60

Source: data/sample1.pdf, Page: 2

QUESTION: What is the product category?
ANSWER: I could not find sufficient information in the provided document.

Source: data/sample1.pdf, Page: 2

QUESTION: What is the contract type?
ANSWER: The contract type is Buy.

Source: data/sample1.pdf, Page: 1

QUESTION: What is the pincode of the tender location?
ANSWER: 629252

Source: data/sample1.pdf, Page: 2

QUESTION: What is being procured and what is its tender value?
ANSWER: Work Item Details
Title
Procurement of LED Street Light
Work Description
Procurement of LED Street Light
NDA/Pre
Qualifi

In [87]:
print(ask_pdf_v3("What is the tender value?"))
print(ask_pdf_v3("What is the product category?"))
print(ask_pdf_v3("What is the contract type?"))

₹14,76,475

Source: data/sample1.pdf
Page: 2
Miscellaneous Goods

Source: data/sample1.pdf
Page: 2
The contract type is Buy.

Source: data/sample1.pdf, Page: 1
